In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)
print("Path:", path)
print(os.listdir(path))

In [ ]:
# Task 1: Write your code here:
paths = os.path.join(path, 'Q1_data.csv')
df =  pd.read_csv(paths)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
plt.hist(df["Delivery_Time"])
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=['Order_ID'])

In [ ]:
df.isna().sum() # Show number of nan in each feature

In [ ]:
# Task 2: Write your code here:
mask = df['Delivery_Time'].notna()
df = df[mask] # drop the missing values at the target


# Check the missing at each column so we able to determine what to do with missing stuff
missing_df = (df.isna().mean().mul(100).round(2).to_frame(name="missing_ratio"))

missing_df["level"] = missing_df["missing_ratio"].apply(
    lambda x: "High" if x >= 55 else "Low"
)

missing_df = missing_df.sort_values(by="missing_ratio", ascending=False)
missing_df

# Fill objects with mode
cols = ["Weather" , "Traffic_Level" ,"Time_of_Day" ]
df[cols] = df[cols].fillna(df[cols].mode().iloc[0])

# FIll numeric with mean
col = ["Courier_Experience_yrs"]
df[col] = df[col].fillna(df[col].mean())


In [ ]:
# Task 3: Write your code here:
df.duplicated().sum() # Show number of duplicates rows
df = df.drop_duplicates()  # Drop duplicates

In [ ]:
# Task 4: Write your code here:
df = pd.get_dummies(df, columns=["Weather" , "Traffic_Level" ,"Time_of_Day" , "Vehicle_Type" ], drop_first=False)



In [ ]:
# Task 5: Write your code here:

from sklearn.preprocessing import StandardScaler
numerical_cols = df.select_dtypes(include=["number"]).columns.drop("Delivery_Time")
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

In [ ]:
# Task 6: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")
#Fine no need to take the log

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=['Delivery_Time'])
y = df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mean_absolute_error
model = RandomForestRegressor(n_estimators=200)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for train_index, test_index in kf.split(X):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Calculate metrics
  results.append(mean_absolute_error(y_test, y_pred))

print(f"MAE: {np.mean(results):.4f}")



In [ ]:
# Task 1: Write your code here:

feature_names = X_train.columns if "X_train" in globals() else X.columns

importance = model.feature_importances_
sorted_imp = sorted(zip(feature_names, importance), key=lambda x: x[1], reverse=True)

features, scores = zip(*sorted_imp)
features = features[:20]
scores = scores[:20]

plt.figure(figsize=(10, 6))
plt.barh(features, scores, color="darkblue")
plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Model Feature Importance")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test)
plt.figure()
plt.scatter(y_train, y_pred)
plt.title("Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.show()

In [ ]:
# Task Bonus: Write your code here: